In [ ]:
import numpy as np
import seaborn as sns
sns.set_theme()

from helper_functions import (sphere_idx, write_text, interpolated_intercepts)

import scipy.constants as constants
import scipy.fft as fft

from matplotlib.ticker import FormatStrFormatter

from skimage.morphology import convex_hull_image

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import h5py
import spimage
import glob
e = constants.elementary_charge
h = constants.Planck
c = constants.speed_of_light

h5File = False
saveFigs = True
multiPRTF = True

filter_errors_custom = False

<h2> Loading and calculating PRTF </h2> 
The best images are selected for alignment/averaging and for the PRTF. 
For libspimage phase retrievals we are looking at the final real space error and select the best subset from this superset. Based on recent results, it seems that the reconstructions might contain two populations. With one subset having a lower final real-space error and the other subset having a higher final real-space error. The first reconstruction sometimes has a weird artefact in the center of the protein. Manually remove this reconstruction from the list and then select 100 best reconstructions. 

In [ ]:
recon_directory = 'phasing_final_check/protein_water_ds_4x_0001_data_1M_prot_wat_3_v_14/'
recon_file = 'phasing_old/protein_water_ds_4x_0001_data_100k_prot_only_0_v_2_h5/phasing_results_v_2.h5'

num_burn_in = 0
if h5File:
    with h5py.File(recon_file, mode='r') as f_ptr:
        real_error = f_ptr['error_real'][num_burn_in:]
        fourier_error = f_ptr['error_fourier'][num_burn_in:]
        support_evolution = f_ptr['support_size'][num_burn_in:]
else:
    real_error = np.load(recon_directory+'error_real.npy')[num_burn_in:]
    fourier_error = np.load(recon_directory+'error_fourier.npy')[num_burn_in:]
    support_evolution = np.load(recon_directory+'support_size.npy')[num_burn_in:]
N_best = real_error.shape[0]

write_text(f'There are {fourier_error.shape[1]} error points!\n')

<h2> Plotting Fourier/Real error </h2>
The first cell will be used to plot the final real-space error for all the N reconstructions and deciding a cut-off point to filter reconstructions further and print some more statistics about this error. The second cell will be the same and show the error in both domains over all reconstructions. I also now plot the evolution of the support size. 

In [ ]:
num_save = 18
num_iters = 900
it_vec = np.linspace(0, num_iters, num_save)

fig_handle = plt.figure(1,constrained_layout = True, dpi = 150) 
fig_handle.patch.set_facecolor(f'white')
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 2)

ax_0 = fig_handle.add_subplot(spec_handle[0,0])
plt.plot(it_vec, fourier_error.T,'o-', ms=3)
plt.yscale('log')
ax_0.set_xticks([0, int(it_vec[-1]/2), int(it_vec[-1])])
ax_0.set_xlabel('iteration #', weight='bold')
ax_0.set_ylabel('error', weight='bold')
ax_0.set_title('Fourier space errors',weight='bold')

ax_1 = fig_handle.add_subplot(spec_handle[0,1])
plt.plot(it_vec, real_error.T,'o-', ms=3)
plt.yscale('log')
ax_1.set_xticks([0, int(it_vec[-1]/2), int(it_vec[-1])])
ax_1.set_xlabel('iteration #', weight='bold')
ax_1.set_title('Real space errors',weight='bold');

if saveFigs:
    plt.savefig(f'figures/'+recon_directory.split(sep='/')[1]+f'_reconstruction_errors_unfiltered.pdf',format='pdf',dpi=200,bbox_inches='tight',pad_inches=0.0);

In [ ]:
fig_handle = plt.figure(2,constrained_layout = True, dpi = 150) 
fig_handle.patch.set_facecolor(f'white')
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 1)

ax_0 = fig_handle.add_subplot(spec_handle[0])
plt.plot(it_vec, support_evolution.T,'o-', ms=3)
plt.yscale('log')
ax_0.set_xticks([0, int(it_vec[-1]/2), int(it_vec[-1])])
ax_0.set_xlabel('iteration #', weight='bold')
ax_0.set_ylabel('size', weight='bold');
ax_0.set_title('Support size evolution',weight='bold');

if saveFigs:
    plt.savefig(f'figures/'+recon_directory.split(sep='/')[1]+f'_support_size_evolution.pdf',format='pdf',dpi=200,bbox_inches='tight',pad_inches=0.0);

The way to do this is just look at the distribution since viewing the y-axis on a log-scale reveals a clear division. Before calculating the PRTF can decide a threshold and filter out these reconstructions. 

In [ ]:
if filter_errors_custom:
    real_error_final = real_error[1:,-1]
    mean_real_error_final = real_error_final.mean()
    median_real_error_final = np.median(real_error_final)
    
    plt.figure(dpi=150)
    plt.plot(real_error_final, 'go', label='_nolegend_')
    plt.axhline(y=mean_real_error_final, xmin=0, xmax=1.0, c='k', ls='--')
    plt.axhline(y=median_real_error_final, xmin=0, xmax=1.0, c='b', ls='--')
    
    plt.xlabel('reconstruction #', weight='bold')
    plt.ylabel('error', weight='bold')
    plt.title(f'Final iteration real-space error distribution over all {N_best} reconstructions', weight='bold');

    cust = 0.488 # v_1: 0.125,0.16,0.09,0.13 and v_2: 0.141,0.185,0.653,0.65,0.54,0.094,0.5,0.385,0.50,0.488
    real_error_final_filter = (real_error_final < cust)
    plt.axhline(y=cust, xmin=0, xmax=1.0, c='r', ls='--')
    plt.legend(['mean','median','custom'], frameon=False, prop={'weight': 'bold'})

    write_text(f'There are {real_error_final_filter.sum()} reconstructions to be included!!!\n')
    write_text(f'There are {N_best - real_error_final_filter.sum()} reconstructions to be filtered out of the reconstructions before merging!!!\n')
    if saveFigs:
        plt.savefig(f'figures/'+recon_directory.split(sep='/')[1]+f'_errors_with_custom_threshold.pdf',format='pdf',dpi=200,bbox_inches='tight',pad_inches=0.0);
else:
    N_best = 100 # 30 for 33 reconstructions / 100 for others reconstructions
    write_text(f'Filtering reconstructions based on {N_best} lowest final real-space error!\n')
    real_error_final_filter = np.argsort(real_error[:, -1])[:N_best]

In [ ]:
if filter_errors_custom:
    write_text('Custom error filtering!\n')
    plt.plot(it_vec, real_error[real_error_final_filter].T,'o-', ms=3)
    plt.xlabel('reconstruction #', weight='bold')
    plt.ylabel('error', weight='bold')
    plt.yscale('log')
    plt.title(f'Filtered real-space errors', weight='bold');
    if saveFigs:
        plt.savefig(f'figures/'+recon_directory.split(sep='/')[1]+f'_reconstruction_errors_filtered.pdf',format='pdf',dpi=200,bbox_inches='tight',pad_inches=0.0);
else:
    write_text('No custom error filtering!\n')
    plt.plot(it_vec, real_error[real_error_final_filter].T,'o-', ms=3)
    plt.xlabel('reconstruction #', weight='bold')
    plt.ylabel('error', weight='bold')
    plt.yscale('log')
    plt.title(f'Filtered real-space errors', weight='bold');
    if saveFigs:
        plt.savefig(f'figures/'+recon_directory.split(sep='/')[1]+f'_reconstruction_errors_filtered.pdf',format='pdf',dpi=200,bbox_inches='tight',pad_inches=0.0);

In [ ]:
write_text('Loading real-space and support data!!!\n')
if h5File:
    with h5py.File(recon_file, mode='r') as f_ptr:
        recon_real = np.squeeze(f_ptr['recon_real'][:])[real_error_final_filter,-1,:,:,:]
        support_arr = np.squeeze(f_ptr['recon_support'][:])[real_error_final_filter,-1,:,:,:]
else:
    recon_real = np.squeeze(np.load(recon_directory+'recon_real.npy'))[:][real_error_final_filter,-1,:,:,:]
    support_arr = np.squeeze(np.load(recon_directory+'recon_support.npy'))[:][real_error_final_filter,-1,:,:,:]

n_rec = support_arr.shape[0]
write_text(f'There are {n_rec} reconstructions\n')

prtf_res = spimage.prtf(recon_real,support_arr,full_out=True)
    
prtf_im = prtf_res['prtf']
prtf_im_abs = np.abs(prtf_im)

s_image = prtf_res['super_image']
s_image_abs = np.abs(s_image)

a_imgs = prtf_res['images']
a_msks = prtf_res['masks']

dimX, dimY, dimZ = a_imgs.shape[1:]
center = dimX // 2

out_directory = f'figures/'+recon_directory.split(sep='/')[1]+f'_{n_rec}_recs'+'_'+'nsp'+'_'
rd = recon_directory.split(sep='/')

<h2> Setting physical constants </h2>
This is particularly important to obtain the correct resolution, since the voxels are dimensionless.
Some parameters are set by the simulations, others are different. For example, after EMC there are more pixels, so 
in principle we have higher resolution, but we won't have any signal there, so in practice this increase is purely a theoretical concept. This increase in array size is purely because of the need to orient the 2D diffraction patterns. Since the pattern might reach further than the array bounds set by simulations/experiments, the array size is increased in order to account for the 
The maximum resolution, or corner resolution is set by the simulation and the PRTF should not reach it.
The below voxel sizes are set by the dimensions of the EMC model, but have otherwise the same parameters as the simulation.

In [ ]:
e_photon_eV = 9000
lambda_photon = (h * c) / (e_photon_eV * e)
d_detector = 0.5
s_pixel = 800e-6
edge_pixel = dimX - dimX//2
write_text(f'Edge pixel: {edge_pixel}\n')

D_particle = 15e-9

pixel_num = dimX - dimX//2
write_text(f'Number of pixels (from center): {pixel_num}\n------------------------------------------------\n')
theta_pixel = 0.5 * np.arctan((pixel_num*s_pixel)/d_detector) 
resolution = lambda_photon/(2.0*np.sin(theta_pixel)) 
pix_real = 0.5 * resolution00
voxel_size = pix_real

write_text(f'X-ray wavelength: {lambda_photon*1e9} nm\n------------------------------------------------\n')
write_text(f'Voxel size (real-space): {voxel_size*1e9} nm\n')
write_text(f'Voxel size (real-space): {voxel_size*1e10} Å\n')

theta_edge = 0.5 * np.arctan((edge_pixel*s_pixel)/d_detector) 
edge_res = lambda_photon/(2.0*np.sin(theta_edge))*1e9
edge_res_inv = 1 / edge_res

write_text(f'Edge resolution: {edge_res} nm\n')
write_text(f'Edge resolution: {edge_res*1e1} Å\n')

center_to_corner = np.sqrt((edge_pixel*s_pixel)**2+(edge_pixel*s_pixel)**2)
theta_corner = 0.5 * np.arctan((center_to_corner)/d_detector)
corner_res = lambda_photon/(2.0 * np.sin(theta_corner))*1e9
corner_res_inv = 1/corner_res

write_text(f'Corner resolution: {corner_res} nm\n')
write_text(f'Corner resolution: {corner_res*1e1} Å\n')

sh_pix_groel = 1 / (D_particle * 1e9) # in nm^-1
pix_emc = 1 / (dimX * voxel_size * 1e9) # in nm^-1
oversampling = sh_pix_groel / pix_emc

write_text(f'Shannon voxel GroEL: {sh_pix_groel} nm^-1\n') 
write_text(f'Voxel size (Fourier-space): {pix_emc} nm^-1\n')
write_text(f'Fourier space oversampling: {oversampling}')

<h2> Plotting maximum projection of the merged density </h2>

In [ ]:
fig_handle = plt.figure(constrained_layout = True, dpi = 220)
fig_handle.patch.set_facecolor(f'white') 
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3) 
cm = 'viridis'

s_image_abs_shifted = fft.fftshift(s_image_abs)

zoom = center
im_slice = center

xy_obj = s_image_abs_shifted.max(axis=0)[im_slice-zoom:im_slice+zoom,im_slice-zoom:im_slice+zoom]
xz_obj = s_image_abs_shifted.max(axis=1)[im_slice-zoom:im_slice+zoom,im_slice-zoom:im_slice+zoom]
yz_obj = s_image_abs_shifted.max(axis=2)[im_slice-zoom:im_slice+zoom,im_slice-zoom:im_slice+zoom]

ax_3 = fig_handle.add_subplot(spec_handle[0])
im_3 = plt.imshow(xy_obj,cmap=cm,interpolation=None)
ax_3.set_xticks([]) 
ax_3.set_yticks([])
minv, maxv = im_3.get_clim()
ax_3.set_title('Maximum projection XY',weight='bold',fontsize=6) 
c_bar_3 = plt.colorbar(im_3,ax=ax_3,fraction=0.001,shrink=0.425,orientation='vertical') 
c_bar_3.set_ticks([minv,maxv])

ax_4 = fig_handle.add_subplot(spec_handle[1]) 
im_4 = plt.imshow(xz_obj,cmap=cm,interpolation=None) 
ax_4.set_xticks([])
ax_4.set_yticks([])
ax_4.set_title('Maximum projection XZ',weight='bold',fontsize=6) 

ax_5 = fig_handle.add_subplot(spec_handle[2]) 
im_5 = plt.imshow(yz_obj,cmap=cm,interpolation=None) 
ax_5.set_xticks([])
ax_5.set_yticks([])
ax_5.set_title('Maximum projection YZ',weight='bold',fontsize=6);

if saveFigs:
    plt.savefig(f'figures/'+recon_directory.split(sep='/')[1]+f'_slice_{im_slice}_avg_dens.pdf',format='pdf',dpi=200,bbox_inches='tight',pad_inches=0.0);
    np.save(f'average_reconstructions/'+rd[1]+f'_dens_{n_rec}r', s_image_abs_shifted)

<h2> Plotting thresholded merged reconstruction </h2> 
This is to obtain the "merged support", since the "super_mask" returned by the Python interface of libspimage isn't
the correct "merged support". The most reliable way is to threshold the amplitude of the merged electron density of GroEL. It seems that for some reconstructions there is quite some sensitivity to the different threshold numbers in how many voxels we keep - for the resolution it doesn't seeem to matter a lot. There is slight shifts when changing the threshold. To compare the thresholded support with the individual reconstructions I also show the total number of points inside the support.

In [ ]:
sup_thresh = 0.10
s_support = s_image_abs > (sup_thresh * s_image_abs.max())
num_vox = a_msks.sum(axis=(1,2,3))
write_text(f'Number of voxels in all {n_rec} individual reconstructions:\n {num_vox}\n')
write_text(f'Number of points in thresholded support: {s_support.sum()}/{np.prod(s_support.shape)}\n')
write_text(f'Fraction of points in thresholded support: {s_support.mean()}\n')
write_text(f'Percentage of points in thresholded support: {s_support.mean()*100}%\n')

s_support = fft.fftshift(s_support)

showSupportMerged = True
if showSupportMerged:
    fig_handle = plt.figure(2,constrained_layout = True, dpi = 170)
    fig_handle.patch.set_facecolor(f'white')  
    spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)
    
    im_slice = center
    
    xy_diff = s_support.max(axis=0)[im_slice-zoom:im_slice+zoom,im_slice-zoom:im_slice+zoom]
    xz_diff = s_support.max(axis=1)[im_slice-zoom:im_slice+zoom,im_slice-zoom:im_slice+zoom]
    yz_diff = s_support.max(axis=2)[im_slice-zoom:im_slice+zoom,im_slice-zoom:im_slice+zoom]

    ax_0 = fig_handle.add_subplot(spec_handle[0,0])
    im_0 = plt.imshow(xy_diff,vmin=0,vmax=1,cmap='gray')
    ax_0.set_xticks([]) 
    ax_0.set_yticks([])
    minv, maxv = im_0.get_clim() 
    ax_0.set_title('Maximum projection XY',weight='bold',fontsize=6) 

    ax_1 = fig_handle.add_subplot(spec_handle[0,1])
    im_1 = plt.imshow(xz_diff,vmin=0,vmax=1,cmap='gray') 
    ax_1.set_xticks([]) 
    ax_1.set_yticks([]) 
    minv, maxv = im_1.get_clim()
    ax_1.set_title('Maximum projection XZ',weight='bold',fontsize=6)

    ax_2 = fig_handle.add_subplot(spec_handle[0,2])
    im_2 = plt.imshow(yz_diff,vmin=0,vmax=1,cmap='gray') 
    ax_2.set_xticks([])
    ax_2.set_yticks([]) 
    minv, maxv = im_2.get_clim() 
    ax_2.set_title('Maximum projection YZ',weight='bold',fontsize=6);
    
if saveFigs:
    plt.savefig(f'figures/'+recon_directory.split(sep='/')[1]+f'_slice_{im_slice}_avg_support_sup_{sup_thresh}.pdf',format='pdf',dpi=200,bbox_inches='tight',pad_inches=0.0);
    np.save(f'average_reconstructions/'+rd[1]+f'_support_{n_rec}r', s_support)

<h2> Visualizing XY, XZ, and YZ slices of PRTF </h2>

In [ ]:
fig_handle = plt.figure(1,constrained_layout = True, dpi = 220)
fig_handle.patch.set_facecolor(f'white') 
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3) 
cm = 'coolwarm' 
max_v = 1e0

im_slice = dimX//2

xy_obj = prtf_im_abs[im_slice,:,:]
xz_obj = prtf_im_abs[:,im_slice,:]
yz_obj = prtf_im_abs[:,:,im_slice]

ax_3 = fig_handle.add_subplot(spec_handle[0,0]) 
im_3 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
ax_3.set_xticks([]) 
ax_3.set_yticks([])
minv, maxv = im_3.get_clim()
ax_3.set_title(f'PRTF (XY-{im_slice}/{dimX})',weight='bold',fontsize=6) 
c_bar_3 = plt.colorbar(im_3, ax=ax_3,fraction=0.01,shrink=0.6,orientation='vertical') 
c_bar_3.set_ticks([minv,maxv]) 

ax_4 = fig_handle.add_subplot(spec_handle[0,1]) 
im_4 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
ax_4.set_xticks([])
ax_4.set_yticks([])
minv, maxv = im_4.get_clim()
ax_4.set_title(f'PRTF (XZ-{im_slice}/{dimY})',weight='bold',fontsize=6) 

ax_5 = fig_handle.add_subplot(spec_handle[0,2]) 
im_5 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
ax_5.set_xticks([]) 
ax_5.set_yticks([]) 
minv, maxv = im_5.get_clim() 
ax_5.set_title(f'PRTF (YZ-{im_slice}/{dimZ})',weight='bold',fontsize=6);

if saveFigs: 
    plt.savefig(f'figures/'+recon_directory.split(sep='/')[1]+f'_slice_{im_slice}_prtf.pdf',format='pdf',dpi=220,bbox_inches='tight',pad_inches=0.0); 

<h2> Radial average of PRTF (3D --> 1D) </h2> 
Before calculating the radial average, we will use a similar method that was used in the past whereby the 3D PRTF was blurred using
a top-hat kernel of size equal to object support. But we use a thresholded support from the merged density. We also take the convex hull to get the shape only without any gaps. 

In [ ]:
conv_hull_supp = convex_hull_image(s_support)

prtf_im_ft = fft.ifftn((prtf_im))
prtf_im_smooth = np.abs(fft.fftn(prtf_im_ft * fft.fftshift(conv_hull_supp)))

num_vox_conv_hull_supp = conv_hull_supp.sum(axis=(0,1,2))
write_text(f'Number of voxels in convex hull of support: {num_vox_conv_hull_supp}\n')

fig_handle = plt.figure(constrained_layout = True, dpi = 220)
fig_handle.patch.set_facecolor(f'white') 
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)
cm = 'coolwarm'
im_slice = center

xy_obj = prtf_im_smooth[im_slice,:,:]
xz_obj = prtf_im_smooth[:,im_slice,:]
yz_obj = prtf_im_smooth[:,:,im_slice]

ax_3 = fig_handle.add_subplot(spec_handle[0])
im_3 = plt.imshow(xy_obj,cmap=cm,interpolation=None)
ax_3.set_xticks([]) 
ax_3.set_yticks([])
minv, maxv = im_3.get_clim()
ax_3.set_title(f'PRTF (XY-{im_slice}/{dimX})',weight='bold',fontsize=6) 
c_bar_3 = plt.colorbar(im_3,ax=ax_3,fraction=0.001,shrink=0.6,orientation='vertical') 
c_bar_3.set_ticks([minv,maxv])

ax_4 = fig_handle.add_subplot(spec_handle[1]) 
im_4 = plt.imshow(xz_obj,cmap=cm,interpolation=None) 
ax_4.set_xticks([])
ax_4.set_yticks([])
ax_4.set_title(f'PRTF (XZ-{im_slice}/{dimY})',weight='bold',fontsize=6) 

ax_5 = fig_handle.add_subplot(spec_handle[2]) 
im_5 = plt.imshow(yz_obj,cmap=cm,interpolation=None) 
ax_5.set_xticks([])
ax_5.set_yticks([])
ax_5.set_title(f'PRTF (YZ-{im_slice}/{dimZ})',weight='bold',fontsize=6);

if saveFigs:
    np.save(f'average_reconstructions/'+rd[1]+'_convex_hull_support', conv_hull_supp)
    plt.savefig(f'figures/'+recon_directory.split(sep='/')[1]+f'_slice_{im_slice}_prtf_blurred.pdf',format='pdf',dpi=220,bbox_inches='tight',pad_inches=0.0);

In [ ]:
plotIP = True

rad_sh = 1 # radius denoting thickness of spherical shell # in voxels
rad_vox = pix_emc

prtf_r = spimage.radial(prtf_im, shell_thickness=rad_sh)
threshold = 1/np.exp(1)

prtf_r_smooth = spimage.radial(prtf_im_smooth, shell_thickness=rad_sh)
prtf_r_smooth /= np.max(prtf_r_smooth)

max_points = prtf_r.shape[0] # maximum number of voxels from PRTF/PRTF_smooth

if rad_sh == 1:
    write_text(f'Size of spherical shell: {rad_sh} voxel(s) or {rad_vox} nm^-1\n')
else:
    rad_vox *= rad_sh
    write_text(f'Size of spherical shell: {rad_sh} voxel(s) or {rad_vox} nm^-1\n') 

# PRTF resolution vector
fp_resolution_r_inv = np.arange(0, max_points) * rad_vox # in nm^-1

xcp, ycp = interpolated_intercepts(fp_resolution_r_inv, prtf_r, np.repeat(threshold, max_points))
xcp_s, ycp_s = interpolated_intercepts(fp_resolution_r_inv, prtf_r_smooth, np.repeat(threshold, max_points))
    
plt.figure(4,dpi=150)
plt.plot(fp_resolution_r_inv, prtf_r, color=mcolors.XKCD_COLORS['xkcd:apple green'], linestyle='--')
plt.plot(fp_resolution_r_inv, prtf_r_smooth, color=mcolors.XKCD_COLORS['xkcd:jungle green'])
plt.axhline(y=threshold, xmin=0, xmax=1.0, c=mcolors.XKCD_COLORS['xkcd:light blue'], linestyle='-')
plt.axvline(x=1/0.6396483204611855, ymin=0 , ymax=1,c='k', linestyle='--', linewidth=1.0, alpha=0.4, label='_nolegend_') # Condor edge-resolution in nm^-1

plt.xlim([0.0, fp_resolution_r_inv[-1]])
plt.ylim([0.0, 1.01])
plt.xlabel('|q| $(nm^{-1})$', weight='bold')
plt.ylabel('PRTF', weight='bold')
plt.legend([f'PRTF ({n_rec})', f'PRTF conv ({n_rec})', '1/e cutoff'], frameon = False, prop=dict(weight='bold', size=8), fontsize=0.5, loc=0)

integral_norm_prtf = np.trapz(prtf_r)
integral_smooth_prtf = np.trapz(prtf_r_smooth)
integral_diff_prtf_abs = np.abs(integral_norm_prtf-integral_smooth_prtf)

if plotIP:
    if (xcp.size != 0) or (xcp_s.size != 0):
        plt.plot(xcp_s, np.repeat(threshold, xcp_s.size),'ko', ms=4.0)
        write_text(f'Intersection(s) blurred PRTF with 1/e threshold: {1/xcp_s} nm\n')
        
if saveFigs:
    with h5py.File(f'figures/'+recon_directory.split(sep='/')[1]+'_PRTF.h5') as prtf_handle:
        prtf_handle['prtf_r_smooth'] = prtf_r_smooth
        prtf_handle['res_r_smooth_nm'] = 1/xcp_s
        prtf_handle['prtf_r'] = prtf_r
        prtf_handle['res_r_nm'] = 1/xcp
        prtf_handle['fp_res_inv_nm'] = fp_resolution_r_inv
        prtf_handle['prtf_3d'] = prtf_im
        prtf_handle['prtf_3d_smooth'] = prtf_im_smooth
        prtf_handle['area_prtf'] = integral_norm_prtf
        prtf_handle['area_prtf_smooth'] = integral_smooth_prtf
        
    plt.savefig(f'figures/'+recon_directory.split(sep='/')[1]+'_PRTF.pdf', format='pdf', dpi=150, bbox_inches='tight', pad_inches=0.0);

write_text(f'{integral_norm_prtf}\n')
write_text(f'{integral_smooth_prtf}\n')
write_text(f'{integral_diff_prtf_abs}\n')

<h2> Plotting multiple PRTFs in single figure </h2> 

In [ ]:
filesPRTF = np.sort(np.array(glob.glob(f'figures_debugging_support/*.h5')))
if multiPRTF:
    names_PRTF = []
    combs_PRTF = []
    res_PRTF = []

    i = 0
    for file in filesPRTF:
        f_name = file.split(sep='/')[1].split(sep='.h5')[0][:-5]
        with h5py.File(file) as f:
            combs_PRTF.append(f['prtf_r_smooth'][:])
            res_PRTF.append(f['fp_res_inv_nm'][:])
            if 'res_r_smooth_nm' in f:
                res_r_smooth_nm = f['res_r_smooth_nm'][()]
                write_text(f'Resolution for [{i}] {f_name}: {res_r_smooth_nm} nm\n')
            else:
                write_text(f'Resolution for [{i}] {f_name}: - nm\n')
        i += 1
        names_PRTF.append(f_name)

    names_PRTF = np.array(names_PRTF)
    combs_PRTF = np.array(combs_PRTF)
    res_PRTF = np.array(res_PRTF)
    max_points = res_PRTF.shape[1]

    res_ax = res_PRTF[0]

    plt.figure(dpi=150)
    plt.axhline(y=1/np.exp(1), xmin=0.0, xmax=res_ax[-1]*10, c='k', linestyle='--', linewidth=1.0, label='_nolegend_') # 1/e threshold
    plt.axvline(x=1/0.6396483204611855, ymin=0 , ymax=1,c='k', linestyle='--', linewidth=1.0, alpha=0.4, label='_nolegend_') # edge resolution simulation
    
    for p in range(len(filesPRTF)):
        plt.plot(res_ax, combs_PRTF[p], linestyle='-')

    plt.xlim([0, res_PRTF[-1][-1]])
    plt.xlabel('|q| $(nm^{-1})$', weight='bold')
    plt.ylabel('PRTF', weight='bold')
    plt.legend(names_PRTF, frameon=True, prop=dict(size=5.0, weight='bold'), loc=1);
    #plt.savefig('ds_4x_prtf_prot_only_debugging_runs.pdf', transparent=False, bbox_inches='tight',dpi=200);